# Minimum dominating set

All fifteen code stages: fresh six-node graph (edge probability 0.6, seed 8), six decision bits, fourteen encoded slack bits, six QAOA layers, penalty 10, 70 optimizer iterations, 70% lower-tail CVaR, optimized and uniform samples, and an independent exhaustive constrained solution.

Reuse the native qfunc for the original seed-8 closed neighborhoods and retain every slack register. Exact local CVaR integrates all joint decision/slack masses. Native Nelder–Mead and an explicit linear pi schedule replace the unpinned hosted optimizer; actual query counts are shown separately from the iteration budget. Use 1000 actual CPU draws for each comparison. NetworkX seeded spring layout renders the same underlying graph with selected nodes and incident edges; it does not reproduce hosted layout coordinates. Exhaustive search over 64 decision assignments replaces Couenne and only runs after sampling. No historical optimizer trajectory or optimum guarantee for the observed quantum sample.

## Follow named decision and slack registers

The preview shows six decision bits, fourteen modeled slack bits and six
cost/mixer layers. The cost helper in main.py declares each closed-neighborhood
constraint and its exact retained slack encoding. FlagQuantum constructs the
coherent phases; zero temporary workspace still leaves twenty modeled bits.
Named-register ordering in workflow.py aligns the joint cost and sampled words.

Try a different twelve-angle binding and check every closed neighborhood
before comparing set size. The 70-iteration CVaR0.7 classical search and its
finite samples do not prove optimality. The bounded exact reference remains
separate. Propagate preview algorithm edits to shared main.py to change the
optimizer's program as well.


## Set up the model and plotting tools

In [ ]:
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    qfunc,
)
from examples.synthesis.applications.optimization.minimum_dominating_set.main import (
    cost,
)


@qfunc
def main(
    params: list[float],
    x: Output[QArray[QBit]],
    s0: Output[QArray[QBit]],
    s1: Output[QArray[QBit]],
    s2: Output[QArray[QBit]],
    s3: Output[QArray[QBit]],
    s4: Output[QArray[QBit]],
    s5: Output[QArray[QBit]],
) -> None:
    allocate(6, x)
    allocate(2, s0)
    allocate(2, s1)
    allocate(3, s2)
    allocate(2, s3)
    allocate(3, s4)
    allocate(2, s5)
    hadamard_transform(x)
    hadamard_transform(s0)
    hadamard_transform(s1)
    hadamard_transform(s2)
    hadamard_transform(s3)
    hadamard_transform(s4)
    hadamard_transform(s5)
    for layer in range(6):
        cost(params[layer], x, s0, s1, s2, s3, s4, s5)
        for j in range(6):
            RX(params[6 + layer], x[j])
        for j in range(2):
            RX(params[6 + layer], s0[j])
        for j in range(2):
            RX(params[6 + layer], s1[j])
        for j in range(3):
            RX(params[6 + layer], s2[j])
        for j in range(2):
            RX(params[6 + layer], s3[j])
        for j in range(3):
            RX(params[6 + layer], s4[j])
        for j in range(2):
            RX(params[6 + layer], s5[j])


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.minimum_dominating_set.overview.model import (
    fresh_model,
)
from examples.synthesis.applications.optimization.minimum_dominating_set.overview.plots import (
    draw_histogram,
    draw_nodes,
    draw_trace,
)
from examples.synthesis.applications.optimization.minimum_dominating_set.overview.workflow import (
    INITIAL,
    SHOTS,
    run_workflow,
)


## Define the domination condition

In [ ]:
model = fresh_model()
print("Every closed neighborhood must intersect the chosen set.")
print("Closed neighborhoods:", model.neighborhoods)


## Generate and draw the original graph

In [ ]:
artifact_dir = Path("artifacts/development/minimum-dominating-set")
display(
    Image(
        filename=str(
            draw_nodes(
                model, artifact_dir / "source-graph.png", "Original six-node graph"
            )
        )
    )
)


## Inspect the constraint encoding

In [ ]:
for neighborhood, coefficients in zip(
    model.neighborhoods, model.slack_weights, strict=True
):
    print(f"sum of choices in {neighborhood} >= 1; slack weights {coefficients}")


## Set the original QAOA workflow dimensions

In [ ]:
print("Six layers, penalty 10, 70 iterations, 70% lower-tail cost")
print("Starting angles:", INITIAL)


## Synthesize the initial native program

In [ ]:
starting_program = synth.synthesize(
    main,
    classical_inputs={"params": INITIAL},
    max_qubits=20,
    max_workspace=0,
    max_operations=4096,
    memory_limit_bytes=64 * 1024**2,
)
print(starting_program.report["resources"])


## Optimize all six layers

In [ ]:
result = run_workflow()
print("Actual objective evaluations:", len(result.trace))
print("Converged:", result.optimizer.converged)
print(
    "Initial and selected CVaR:",
    result.trace[0].cost,
    min(r.cost for r in result.trace),
)


## Read the actual cost trace

In [ ]:
display(Image(filename=str(draw_trace(result, artifact_dir / "cost-trace.png"))))


## Inspect the five lowest observed penalized costs

In [ ]:
for row in result.optimized[:5]:
    print(row)


## Check the uniform comparison draws

In [ ]:
print(
    "Uniform and optimized draws:",
    sum(row.count for row in result.uniform),
    sum(row.count for row in result.optimized),
)
print("Configured draws per comparison:", SHOTS)


## Compare the observed distributions

In [ ]:
display(
    Image(filename=str(draw_histogram(result, artifact_dir / "cost-histogram.png")))
)


## Select the best actually observed joint assignment

In [ ]:
best = result.optimized[0]
print("Best actually observed:", best)


## Draw the observed set

In [ ]:
display(
    Image(
        filename=str(
            draw_nodes(
                model,
                artifact_dir / "observed-set.png",
                f"Observed cost {best.cost}, feasible={best.feasible}",
                best.decisions,
            )
        )
    )
)


## Read the independent exact constrained solution

In [ ]:
print("All exact minimum decision words:", result.classical)
print("Minimum set size:", result.classical[0].bit_count())


## Draw one exact set

In [ ]:
display(
    Image(
        filename=str(
            draw_nodes(
                model,
                artifact_dir / "exact-set.png",
                "One exact minimum dominating set",
                result.classical[0],
            )
        )
    )
)
